# Notebook B: from-scratch DPO (one dataset per notebook)
**Settings:** Accelerator = GPU T4 x2, Internet = OFF is fine.
**Add data:** `stf-backup` (your SFT run) and `dpo-data` (made by Notebook A).
1. Run once with `SMOKE_TEST = True` and check the output and seconds per step.
2. Set `SMOKE_TEST = False`, then Save Version -> **Save & Run All (Commit)**.
3. For the second run: File -> Save a copy, change only `DATASET_NAME` to `ultrafeedback`.

In [ ]:
import os, json, math, time, random, shutil
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


DATASET_NAME = "hh_rlhf"      # "hh_rlhf" or "ultrafeedback"
SMOKE_TEST   = False           


BETA, LR = 0.1, 1e-6
MAX_LENGTH, MAX_PROMPT_LEN = 512, 256
GRAD_ACCUM, CLIP = 32, 1.0            # per-step batch = 1 pair, 32 pairs per update
TRAIN_PAIRS, VAL_PAIRS = 15_000, 1_000
EVAL_EVERY = 100                      # optimizer steps between full validations
SEED = 42
USE_GRAD_CKPT = False                 
POLICY_DEV, REF_DEV = "cuda:0", "cuda:1"

assert torch.cuda.device_count() >= 2, "Choose Accelerator = GPU T4 x2"
OUT = Path(f"/kaggle/working/dpo_{DATASET_NAME}" + ("_smoke" if SMOKE_TEST else ""))
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Run:", DATASET_NAME, "| smoke test:", SMOKE_TEST, "| output:", OUT)

In [ ]:
def find_sft():
    for root, _, files in os.walk("/kaggle/input"):
        if "model.safetensors" in files and root.replace("\\", "/").endswith("sft_run/best_checkpoint"):
            return Path(root)
    raise FileNotFoundError("SFT best_checkpoint not found: attach the stf-backup dataset")

def find_file(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files and (Path(root) / "tokenizer").is_dir():
            return Path(root) / name
    raise FileNotFoundError(f"{name} not found: attach the dpo-data dataset")

SFT_DIR = find_sft()
train_path = find_file(f"{DATASET_NAME}_train.json")
val_path = find_file(f"{DATASET_NAME}_validation.json")

tok = AutoTokenizer.from_pretrained(train_path.parent / "tokenizer")
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

train_rows = json.load(open(train_path))[:TRAIN_PAIRS]
val_rows = json.load(open(val_path))[:VAL_PAIRS]
if SMOKE_TEST:
    train_rows = train_rows[:GRAD_ACCUM * 3]     # 3 optimizer steps
    val_rows = val_rows[:40]
    EVAL_EVERY = 2

print("SFT checkpoint:", SFT_DIR)
print("Train pairs:", len(train_rows), "| Val pairs:", len(val_rows))

In [ ]:
def build_examples(rows):
    """Tokenize once. Prompt keeps its END if too long; response gets EOS and is truncated to fit."""
    P = tok([r["prompt"] for r in rows], add_special_tokens=False)["input_ids"]
    C = tok([r["chosen"] for r in rows], add_special_tokens=False)["input_ids"]
    R = tok([r["rejected"] for r in rows], add_special_tokens=False)["input_ids"]
    out, cut_prompt, cut_resp = [], 0, 0
    for p, c, r in zip(P, C, R):
        if len(p) > MAX_PROMPT_LEN:
            p = p[-MAX_PROMPT_LEN:]; cut_prompt += 1
        room = MAX_LENGTH - len(p) - 1                  # 1 slot for EOS
        if len(c) > room or len(r) > room:
            cut_resp += 1
        out.append((p, c[:room] + [tok.eos_token_id], r[:room] + [tok.eos_token_id]))
    print(f"  prompts truncated: {cut_prompt}/{len(rows)} | pairs with truncated response: {cut_resp}/{len(rows)}")
    return out

print("Tokenizing train..."); train_ex = build_examples(train_rows)
print("Tokenizing val...");   val_ex = build_examples(val_rows)

In [ ]:
def collate(seqs):
    """seqs = [(prompt_ids, response_ids), ...] -> padded ids, attention mask, labels.
    Labels are -100 on prompt and padding, so only response tokens are scored."""
    L = max(len(p) + len(r) for p, r in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    lab = torch.full((len(seqs), L), -100, dtype=torch.long)
    for i, (p, r) in enumerate(seqs):
        n = len(p) + len(r)
        ids[i, :n] = torch.tensor(p + r)
        att[i, :n] = 1
        lab[i, len(p):n] = torch.tensor(r)
    return ids, att, lab

def seq_logps(model, ids, att, lab, chunk=1024):
    """Sum of response-token log-probs for each row. Only response positions go through lm_head."""
    dev = next(model.parameters()).device
    ids, att, lab = ids.to(dev), att.to(dev), lab.to(dev)
    hidden = model.model(input_ids=ids, attention_mask=att, use_cache=False).last_hidden_state
    h, tgt = hidden[:, :-1], lab[:, 1:]                 # position t-1 predicts token t
    mask = tgt.ne(-100)
    sel_h, sel_t = h[mask], tgt[mask]
    seq_id = torch.arange(ids.size(0), device=dev).unsqueeze(1).expand_as(tgt)[mask]
    parts = []
    for s in range(0, sel_t.numel(), chunk):
        logits = model.lm_head(sel_h[s:s + chunk]).float()
        parts.append(-F.cross_entropy(logits, sel_t[s:s + chunk], reduction="none"))
    token_lp = torch.cat(parts)
    return torch.zeros(ids.size(0), device=dev).index_add(0, seq_id, token_lp)

def dpo_terms(pc, pr, rc, rr):
    """margin = r_w - r_l = beta * [(pi_w - ref_w) - (pi_l - ref_l)];  loss = -log sigmoid(margin)."""
    margin = BETA * ((pc - rc) - (pr - rr))
    return -F.logsigmoid(margin), margin

In [ ]:

ref = AutoModelForCausalLM.from_pretrained(str(SFT_DIR), torch_dtype=torch.float16).to(REF_DEV)
ref.eval()
for p_ in ref.parameters():
    p_.requires_grad_(False)

@torch.no_grad()
def precompute_ref(examples, bs=8):
    seqs = []
    for i, (p, c, r) in enumerate(examples):
        seqs.append((i, 0, p, c)); seqs.append((i, 1, p, r))
    order = sorted(range(len(seqs)), key=lambda k: len(seqs[k][2]) + len(seqs[k][3]))
    out = torch.zeros(len(examples), 2)
    t0 = time.time()
    for n, s in enumerate(range(0, len(order), bs)):
        batch = [seqs[k] for k in order[s:s + bs]]
        ids, att, lab = collate([(b[2], b[3]) for b in batch])
        lp = seq_logps(ref, ids, att, lab).float().cpu()
        for (i, kind, _, _), v in zip(batch, lp):
            out[i, kind] = v
        if n % 500 == 0:
            print(f"  reference {s}/{len(order)} sequences | {time.time()-t0:.0f}s", flush=True)
    return out

print("Reference log-probs: train"); ref_train = precompute_ref(train_ex)
print("Reference log-probs: val");   ref_val = precompute_ref(val_ex)
torch.save({"train": ref_train, "val": ref_val}, OUT / "ref_logps.pt")

del ref
torch.cuda.empty_cache()
print("Reference done and freed. Example chosen/rejected ref logps:", ref_train[0].tolist())

In [ ]:
policy = AutoModelForCausalLM.from_pretrained(str(SFT_DIR), torch_dtype=torch.float32).to(POLICY_DEV)
policy.config.use_cache = False
if USE_GRAD_CKPT:
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

optimizer = torch.optim.RMSprop(policy.parameters(), lr=LR)       # fixed LR, no schedule
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)           # fp16 autocast + fp32 weights

def pair_logps(ex):
    p, c, r = ex
    ids, att, lab = collate([(p, c), (p, r)])                      # chosen and rejected as a batch of 2
    with torch.autocast("cuda", dtype=torch.float16):
        lp = seq_logps(policy, ids, att, lab)
    return lp[0], lp[1]

@torch.no_grad()
def evaluate(examples, ref_lp, label):
    policy.eval()
    losses, margins = [], []
    for i, ex in enumerate(examples):
        pc, pr = pair_logps(ex)
        loss, m = dpo_terms(pc, pr, ref_lp[i, 0].to(POLICY_DEV), ref_lp[i, 1].to(POLICY_DEV))
        losses.append(loss.item()); margins.append(m.item())
    policy.train()
    res = {"val_dpo_loss": sum(losses) / len(losses),
           "val_reward_acc": sum(m > 0 for m in margins) / len(margins),   # strict r_w > r_l
           "val_reward_margin": sum(margins) / len(margins),
           "num_pairs": len(margins)}
    print(f"{label}: loss={res['val_dpo_loss']:.4f} acc={res['val_reward_acc']:.4f} margin={res['val_reward_margin']:.4f}", flush=True)
    return res

step0 = evaluate(val_ex, ref_val, "Step 0 (policy == reference; margin should be ~0)")

In [ ]:
def save_ckpt(path, metrics):
    if path.exists():
        shutil.rmtree(path)
    policy.save_pretrained(path, safe_serialization=True)
    tok.save_pretrained(path)
    json.dump(metrics, open(path / "metrics.json", "w"), indent=2)
    print("Saved:", path, flush=True)

run_cfg = dict(dataset=DATASET_NAME, beta=BETA, lr=LR, optimizer="RMSprop", grad_accum=GRAD_ACCUM,
               clip=CLIP, max_length=MAX_LENGTH, max_prompt_len=MAX_PROMPT_LEN, seed=SEED,
               train_pairs=len(train_ex), val_pairs=len(val_ex), eval_every=EVAL_EVERY,
               precision="fp32 weights + fp16 autocast + GradScaler; reference fp16",
               prompt_format="tokenizer.apply_chat_template(user/assistant turns, add_generation_prompt=True), same as SFT")
json.dump(run_cfg, open(OUT / "run_config.json", "w"), indent=2)

order = list(range(len(train_ex)))
random.Random(SEED).shuffle(order)
total_steps = math.ceil(len(order) / GRAD_ACCUM)
history = [{"step": 0, "type": "val", **step0}]
best_acc, skipped = -1.0, 0
t0 = time.time()
policy.train()

for step in range(1, total_steps + 1):
    idxs = order[(step - 1) * GRAD_ACCUM: step * GRAD_ACCUM]
    optimizer.zero_grad(set_to_none=True)
    losses, accs = [], []
    for i in idxs:
        pc, pr = pair_logps(train_ex[i])
        loss, m = dpo_terms(pc, pr, ref_train[i, 0].to(POLICY_DEV), ref_train[i, 1].to(POLICY_DEV))
        scaler.scale(loss / len(idxs)).backward()               # accumulate: mean over the group
        losses.append(loss.item()); accs.append(float(m.item() > 0))
    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(policy.parameters(), CLIP).item()
    before = scaler.get_scale()
    scaler.step(optimizer); scaler.update()
    skipped += int(scaler.get_scale() < before)                 # fp16 overflow -> step skipped

    if step % 5 == 0 or step == 1 or step == total_steps:
        el = time.time() - t0
        print(f"step {step}/{total_steps} | loss {sum(losses)/len(losses):.4f} | "
              f"train acc {sum(accs)/len(accs):.2f} | grad norm {gnorm:.1f} | skipped {skipped} | "
              f"{el/step:.1f}s/step | ETA {el/step*(total_steps-step)/60:.0f} min", flush=True)

    if step % EVAL_EVERY == 0 or step == total_steps:
        res = evaluate(val_ex, ref_val, f"Step {step} full validation")
        history.append({"step": step, "type": "val", **res})
        if res["val_reward_acc"] > best_acc:                    # best = highest validation reward accuracy
            best_acc = res["val_reward_acc"]
            save_ckpt(OUT / "best_checkpoint", {"step": step, **res})
        json.dump(history, open(OUT / "final_metrics.json", "w"), indent=2)

save_ckpt(OUT / "final_checkpoint", history[-1])
json.dump(history, open(OUT / "final_metrics.json", "w"), indent=2)
print(f"Done in {(time.time()-t0)/60:.0f} min. Best val reward accuracy: {best_acc:.4f}")

In [ ]:
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT), f"{p.stat().st_size/1e6:.1f} MB")
print(json.dumps(history, indent=2)[:3000])